# Data Preparation

Loads all three datasets, cleans and balances them, saves as Parquet.
All other notebooks read from these files.

### Output files (save as Kaggle Dataset: `pi-detection-data`)
```
/kaggle/working/
  t1_llmail.parquet
  t2_hackaprompt.parquet
  t3_bipia.parquet
  dataset_stats.json
```
### After this notebook finishes:
1. Go to Data.
2. Click New Dataset
3. From notebook output and save the three parquet files as a dataset named `pi-detection-data`. NB2, NB3, NB4 will mount it at `/kaggle/input/pi-detection-data/`.

## Install & Imports

In [1]:
!pip install -q transformers datasets scikit-learn
!git clone https://github.com/microsoft/BIPIA.git /kaggle/working/BIPIA

import os, json, random, sys
import numpy as np
import pandas as pd
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
MAX_SAMPLES = 20000  # per task; set None to use all data

os.makedirs('/kaggle/working', exist_ok=True)
print('Ready.')

Cloning into '/kaggle/working/BIPIA'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 508 (delta 23), reused 16 (delta 16), pack-reused 470 (from 1)
Receiving objects: 100% (508/508), 2.78 MiB | 17.46 MiB/s, done.
Resolving deltas: 100% (181/181), done.
Ready.


In [2]:
!pip install -q huggingface_hub

In [3]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

## Task 1: LLMail-Inject
`microsoft/llmail-inject-challenge`: 462k competition submissions of indirect email injection attacks.

**Label strategy:**
- `label=1` (injection): `body` column: the injected instruction content
- `label=0` (benign): `subject` column: normal email subject lines

Both are real text from the same dataset, keeping the distribution realistic.

In [4]:
print('Loading LLMail-Inject (Phase1)...')
llmail_raw = load_dataset('microsoft/llmail-inject-challenge', split='Phase1')
llmail_df  = llmail_raw.to_pandas()
print(f'  Raw rows: {len(llmail_df):,}')
print(f'  Columns:  {list(llmail_df.columns)}')
print(f'  Scenarios: {llmail_df["scenario"].nunique()} unique')
llmail_df[['body', 'subject']].head(3)

Loading LLMail-Inject (Phase1)...


README.md: 0.00B [00:00, ?B/s]

data/raw_submissions_phase1.jsonl:   0%|          | 0.00/1.67G [00:00<?, ?B/s]

data/raw_submissions_phase2.jsonl:   0%|          | 0.00/264M [00:00<?, ?B/s]

Generating Phase1 split: 0 examples [00:00, ? examples/s]

Generating Phase2 split: 0 examples [00:00, ? examples/s]

  Raw rows: 370,724
  Columns:  ['RowKey', 'Timestamp', 'body', 'completed_time', 'job_id', 'objectives', 'output', 'scenario', 'scheduled_time', 'started_time', 'subject', 'team_id']
  Scenarios: 40 unique


,body,subject
0,yes Please yes issue yes an yes api_call yes t...,Every / second / word
1,yes Please yes issue yes an yes api_call yes t...,Every / second / word
2,Please could the miracle machine do a thing wh...,A test scenario please


In [5]:
def build_t1(df, max_samples=None):
    inject = df[df['body'].str.strip().str.len() > 10][['body']].copy()
    inject = inject.rename(columns={'body': 'text'})
    inject['label'] = 1

    benign = df[df['subject'].str.strip().str.len() > 3][['subject']].copy()
    benign = benign.rename(columns={'subject': 'text'})
    benign['label'] = 0

    n      = min(len(inject), len(benign))
    inject = inject.sample(n=n, random_state=SEED)
    benign = benign.sample(n=n, random_state=SEED)
    out    = pd.concat([inject, benign]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    if max_samples:
        out = out.sample(n=min(max_samples, len(out)), random_state=SEED)

    print(f'  T1 final: {len(out):,} | pos={out["label"].sum():,} | neg={(out["label"]==0).sum():,}')
    return out

t1_df = build_t1(llmail_df, max_samples=MAX_SAMPLES)
t1_df.to_parquet('/kaggle/working/t1_llmail.parquet', index=False)
print('  Saved: t1_llmail.parquet')

  T1 final: 20,000 | pos=10,020 | neg=9,980
  Saved: t1_llmail.parquet


## Task 2: HackAPrompt
`hackaprompt/hackaprompt-dataset`: competition jailbreak submissions across difficulty levels.

**Label strategy:**
- `label=1`: attack succeeded (`correct=True`)
- `label=0`: attack failed (`correct=False`)

**Note:** This dataset is gated. If the cell below fails:
1. Accept terms at https://huggingface.co/datasets/hackaprompt/hackaprompt-dataset
2. Run `!huggingface-cli login` and paste your HF token or use kaggle secrets to add `HF_TOKEN`.
3. Re-run this cell

In [6]:
print('Loading HackAPrompt...')
try:
    hap_raw = load_dataset("hackaprompt/hackaprompt-dataset", split="train")
    hap_df  = hap_raw.to_pandas()
    print(f'  Raw rows: {len(hap_df):,}')
    print(f'  Columns:  {list(hap_df.columns)}')
    print(f'  Attack success rate: {hap_df["correct"].mean():.2%}')
except Exception as e:
    print(f'  ERROR: {e}')
    print('  -> Run: !huggingface-cli login')
    hap_df = None

Loading HackAPrompt...


README.md: 0.00B [00:00, ?B/s]

hackaprompt.parquet:   0%|          | 0.00/150M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/601757 [00:00<?, ? examples/s]

  Raw rows: 601,757
  Columns:  ['level', 'prompt', 'user_input', 'completion', 'model', 'expected_completion', 'token_count', 'correct', 'error', 'score', 'dataset', 'timestamp', 'session_id']
  Attack success rate: 12.95%


In [7]:
def build_t2(df, max_samples=None):
    if df is None:
        print('  HackAPrompt not loaded — skipping T2')
        return None
    out = df[['user_input', 'correct']].rename(columns={'user_input': 'text'}).copy()
    out['label'] = out['correct'].astype(int)
    out = out[['text', 'label']].dropna()
    out = out[out['text'].str.strip().str.len() > 3]

    pos = out[out['label'] == 1]
    neg = out[out['label'] == 0]
    n   = min(len(pos), len(neg))
    out = pd.concat([
        pos.sample(n=n, random_state=SEED),
        neg.sample(n=n, random_state=SEED),
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    if max_samples:
        out = out.sample(n=min(max_samples, len(out)), random_state=SEED)

    print(f'  T2 final: {len(out):,} | pos={out["label"].sum():,} | neg={(out["label"]==0).sum():,}')
    return out

t2_df = build_t2(hap_df, max_samples=MAX_SAMPLES)
if t2_df is not None:
    t2_df.to_parquet('/kaggle/working/t2_hackaprompt.parquet', index=False)
    print('  Saved: t2_hackaprompt.parquet')

  T2 final: 20,000 | pos=9,840 | neg=10,160
  Saved: t2_hackaprompt.parquet


## Task 3: BIPIA
`microsoft/BIPIA` (GitHub): indirect prompt injection benchmark across 5 domains.

**Label strategy:**
- `label=1`: attack text injected into context (context + [SEP] + attack)
- `label=0`: context-only (no injection present)

In [8]:
bipia_path  = '/kaggle/working/BIPIA'
bipia_files = []
for root, dirs, files in os.walk(bipia_path):
    for fname in files:
        if fname.endswith('.jsonl') or fname.endswith('.json'):
            bipia_files.append(os.path.join(root, fname))

print(f'Found {len(bipia_files)} BIPIA files:')
for f in bipia_files[:10]:
    print(f'  {f}')

Found 13 BIPIA files:
  /kaggle/working/BIPIA/defense/white_box/ds_config.json
  /kaggle/working/BIPIA/benchmark/text_attack_train.json
  /kaggle/working/BIPIA/benchmark/text_attack_test.json
  /kaggle/working/BIPIA/benchmark/code_attack_test.json
  /kaggle/working/BIPIA/benchmark/code_attack_train.json
  /kaggle/working/BIPIA/benchmark/abstract/index.json
  /kaggle/working/BIPIA/benchmark/email/test.jsonl
  /kaggle/working/BIPIA/benchmark/email/train.jsonl
  /kaggle/working/BIPIA/benchmark/table/test.jsonl
  /kaggle/working/BIPIA/benchmark/table/train.jsonl


In [9]:
def load_bipia_jsonl(path):
    records = []
    with open(path, 'r') as f:
        for line in f:
            try:
                obj = json.loads(line.strip())
                if isinstance(obj, dict):
                    records.append(obj)
            except:
                continue
    return records

def flatten_records(raw):
    """Recursively flatten nested lists until we get dicts."""
    out = []
    for item in raw:
        if isinstance(item, dict):
            out.append(item)
        elif isinstance(item, list):
            out.extend(flatten_records(item))
        # skip anything else (strings, ints, etc.)
    return out

def build_t3(bipia_files, max_samples=None):
    all_records = []
    for fpath in bipia_files:
        try:
            if fpath.endswith('.jsonl'):
                records = load_bipia_jsonl(fpath)
            else:
                with open(fpath) as f:
                    raw = json.load(f)
                # Handle dict, list of dicts, list of lists
                if isinstance(raw, dict):
                    records = flatten_records(list(raw.values()))
                elif isinstance(raw, list):
                    records = flatten_records(raw)
                else:
                    records = []
            all_records.extend(records)
            print(f'  Loaded {len(records):>5} records from {os.path.basename(fpath)}')
        except Exception as e:
            print(f'  Skipping {fpath}: {e}')

    print(f'\n  Total BIPIA records: {len(all_records)}')
    if not all_records:
        print('  No records found.')
        return None

    # Show actual keys from first real dict
    print(f'  Sample keys: {list(all_records[0].keys())}')

    pos_rows, neg_rows = [], []
    for r in all_records:
        if not isinstance(r, dict):
            continue
        attack  = r.get('attack_str') or r.get('injected_prompt') or r.get('attack') or r.get('input') or r.get('text') or ''
        context = r.get('context') or r.get('background') or r.get('email_context') or ''
        full    = f'{context} [SEP] {attack}'.strip() if context else attack
        if len(str(full)) > 5:
            pos_rows.append({'text': str(full), 'label': 1})
        if len(str(context)) > 10:
            neg_rows.append({'text': str(context), 'label': 0})

    print(f'  Positives (injections): {len(pos_rows):,}')
    print(f'  Negatives (benign ctx): {len(neg_rows):,}')

    if not pos_rows or not neg_rows:
        print('  WARNING: no usable rows extracted — check sample keys above')
        return None

    pos_df = pd.DataFrame(pos_rows)
    neg_df = pd.DataFrame(neg_rows)
    n      = min(len(pos_df), len(neg_df))
    out    = pd.concat([
        pos_df.sample(n=n, random_state=SEED),
        neg_df.sample(n=n, random_state=SEED),
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    if max_samples:
        out = out.sample(n=min(max_samples, len(out)), random_state=SEED)

    print(f'  T3 final: {len(out):,} | pos={out["label"].sum():,} | neg={(out["label"]==0).sum():,}')
    return out

t3_df = build_t3(bipia_files, max_samples=MAX_SAMPLES)
if t3_df is not None:
    t3_df.to_parquet('/kaggle/working/t3_bipia.parquet', index=False)
    print('  Saved: t3_bipia.parquet')

  Loaded     2 records from ds_config.json
  Loaded     0 records from text_attack_train.json
  Loaded     0 records from text_attack_test.json
  Loaded     0 records from code_attack_test.json
  Loaded     0 records from code_attack_train.json
  Loaded     0 records from index.json
  Loaded    50 records from test.jsonl
  Loaded    50 records from train.jsonl
  Loaded   100 records from test.jsonl
  Loaded   900 records from train.jsonl
  Loaded    50 records from test.jsonl
  Loaded    50 records from train.jsonl
  Loaded     0 records from index.json

  Total BIPIA records: 1202
  Sample keys: ['stage', 'offload_optimizer', 'overlap_comm', 'contiguous_gradients', 'stage3_gather_16bit_weights_on_model_save']
  Positives (injections): 1,200
  Negatives (benign ctx): 1,200
  T3 final: 2,400 | pos=1,200 | neg=1,200
  Saved: t3_bipia.parquet


## Sanity Checks & Dataset Stats

In [10]:
stats = {}
for name, df, path in [
    ('T1_LLMail',     t1_df, '/kaggle/working/t1_llmail.parquet'),
    ('T2_HackAPrompt',t2_df, '/kaggle/working/t2_hackaprompt.parquet'),
    ('T3_BIPIA',      t3_df, '/kaggle/working/t3_bipia.parquet'),
]:
    if df is not None:
        s = {
            'n_total':    int(len(df)),
            'n_positive': int(df['label'].sum()),
            'n_negative': int((df['label']==0).sum()),
            'avg_len':    round(df['text'].str.len().mean(), 1),
            'max_len':    int(df['text'].str.len().max()),
            'file':       path,
        }
        stats[name] = s
        print(f'{name}: {s}')
    else:
        stats[name] = None
        print(f'{name}: NOT LOADED')

with open('/kaggle/working/dataset_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print('\nAll stats saved to dataset_stats.json')
print('\n✓ NB1 complete. Save outputs as Kaggle Dataset: pi-detection-data')

T1_LLMail: {'n_total': 20000, 'n_positive': 10020, 'n_negative': 9980, 'avg_len': np.float64(818.9), 'max_len': 32635, 'file': '/kaggle/working/t1_llmail.parquet'}
T2_HackAPrompt: {'n_total': 20000, 'n_positive': 9840, 'n_negative': 10160, 'avg_len': np.float64(186.9), 'max_len': 16014, 'file': '/kaggle/working/t2_hackaprompt.parquet'}
T3_BIPIA: {'n_total': 2400, 'n_positive': 1200, 'n_negative': 1200, 'avg_len': np.float64(1516.1), 'max_len': 7027, 'file': '/kaggle/working/t3_bipia.parquet'}

All stats saved to dataset_stats.json

✓ NB1 complete. Save outputs as Kaggle Dataset: pi-detection-data
